# **MEGA Workshop Part 1: Extraction**

Hello! Welcome to the first segment of this workshop! Here, we will cover the basics of web scraping on a static website. From this segment, you will learn how to:
1. Send a HTTP GET request to a URL
2. Parse the static HTML
3. Find relevant data from raw HTML

The websites used for web scraping are https://www.scrapethissite.com/pages/simple/ and https://books.toscrape.com/.

Let's begin!

## Install necessary dependencies

In [ ]:
!pip install beautifulsoup4
!pip install requests

## Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import random

## Send a HTTP GET request to a URL



In [ ]:
# By default, the requests library identifies itself as 'User-Agent': 'python-requests/2.31.0'
# Most web servers block this immediately. We want to "mask" our script to look more like a browser.
# We can copy the headers used in our actual browser.
# You might still get blocked if you're constantly on the same IP address.

HEADERS = {"Accept": "*/*",
           "Accept-Encoding": "gzip, deflate, br, zstd",
           "Accept-Language": "en-GB,en-US;q=0.9,en;q=0.8,zh-CN;q=0.7,zh-TW;q=0.6,zh;q=0.5",
           "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
          }

# Get HTML of the page
url = "https://www.scrapethissite.com/pages/simple/"
response = requests.get(url, headers = HEADERS)

# Check that the GET request is successful
print(f"Status Code: {response.status_code}")
print(f"URL: {response.url}")

**Status Codes**

A server will always include a numeric status code in the response message.
There are 70 status codes, grouped into
1. 1XX: informational responses
2. 2XX: successful responses
3. 3XX: redirection messages
4. 4XX: client error messages
5. 5XX: server error messages

Here are some frequently used status codes:
- "200 OK" -- server has correctly processed the request
- "404 Not Found" -- resource we are requesting does not exist
- "500 Internal Server Error" -- an error occurred in the server
- "406 Not Acceptable" -- server cannot reply in the requested format e.g. JSON
- "429 Too Many Requests" -- too many requests sent by client

Some other status codes:
- "201 Created" -- Specific to POST operation when an object is created
- "202 Accepted" -- Noncommittal. Indicates that a job has been accepted
- "204 No Content" -- Success! Often means an object has been deleted or there is otherwise no useful info to send back
- "301 Moved Permanently" -- URL changed permanently
- "400 Bad Request" -- Client error

### Validation Checks

There is more to the response object. We can conduct a short investigation using some of its methods.

**Imagine this as checking the label of a package before opening it.**

In [ ]:
# Identifies the software used by the web server to handle your request
print(f"Server Type: {response.headers.get('Server')}")

# Possible Outputs
# (1) nginx
# (2) Apache
# (3) Microsoft-IIS

# Since we see cloudflare, the site has strong anti-bot protections.
# If the scraper fails, it ix likely because cloudflare is blocking you.

In [ ]:
# Tells you what kind of file the server handled over
print(f"Content Type: {response.headers.get('Content-Type')}")

# Usually returns something like
# (1) text/html; charset=UTF-8
# (2) application/json

You can also check the response time to see how long the extraction took. In Data Engineering, speed matters. Knowing the average time per page can help you estimate the total runtime of your pipeline.



Using this method can also allow you to check if the server is intentially delaying its response to waste your execution time and make scraping unviable.

In [ ]:
# Time taken to fetch data
print(f"Time taken to fetch data: {response.elapsed.total_seconds()} seconds")

For debugging purposes, you can also check what was sent to the server.

In [ ]:
# This object shows exactly what you sent to the server
print(response.request)

# Check if "User-Agent" was actually included in the request
print(response.request.headers)

### Check the response text

In [ ]:
# Access the response's content as text. We should get the html text.
print(f"Response Text: {response.text}")

## Parse the static HTML

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# 2nd argument can also take in
# (1) "lxml": fast, lenient parser built on C (pip install lxml)
# (2) "html5lib": extremely lenient parser, useful for messy HTML (pip install html5lib)

## Find relevant data

The class attribute is a space-separated list so the browser & BeautifulSoup see 2 distinct classes assigned to this div element, "col-md-4" and "country". "country-name" is treated as a single, unique identifier even though it has the word "country" in it.

In [ ]:
# Find all the div containers for countries
# Want: the text in <div class="col-md-4 country">
country_elements = soup.find_all("div", class_ = "country")

# Returns a list-like object so you can use indexing.
print(type(country_elements))
print()
print(country_elements[0])

Find web links

In [ ]:
# Find all anchor tags with href attribute
links = soup.find_all("a", href=True)
print(links[6])

for link in links:
  print(f"Found URL: {link["href"]}")

### Prepare data for transformation

In [ ]:
# We want to handle missing values that could crash the script when we call .text
countries_data = []

for item in country_elements:
  name_tag = item.find("h3", class_ = "country-name")
  capital_tag = item.find("span", class_ = "country-capital")
  population_tag = item.find("span", class_ = "country-population")
  area_tag = item.find("span", class_ = "country-area")

  # Extract text only if it exists
  name = name_tag.text.strip() if name_tag else "Unknown Name"
  capital = capital_tag.text.strip() if capital_tag else "N/A"
  population = population_tag.text.strip() if population_tag else "N/A"
  area = area_tag.text.strip() if area_tag else "N/A"

  countries_data.append({
      "Country": name,
      "Metadata": {
          "Capital": capital,
          "Population": population,
          "Area (km^2)" : area
      }
  })

# Show the first entry in the list
print(countries_data[0])

### Export data

We will output the extracted data in JSON format for the transformation team.

In [ ]:
with open("scraped_data.json", "w") as file:
  json.dump(countries_data, file)

## Example 2

In [ ]:
headers = {"Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
           "Accept-Encoding": "gzip, deflate, br, zstd",
           "Accept-Language": "en-GB,en-US;q=0.9,en;q=0.8,zh-CN;q=0.7,zh-TW;q=0.6,zh;q=0.5",
           "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
          }

def extract_book_info(page_number=1):
  base_url = f"https://books.toscrape.com/catalogue/page-{page_number}.html"
  response = requests.get(base_url, headers = headers)

  # Validation Checks
  if page_number == 1:
    print("-------- Validation Checks --------")
    print(f"Status Code: {response.status_code}")
    print(f"URL: {response.url}")
    print(f"Server Type: {response.headers.get('Server')}")
    print(f"Content Type: {response.headers.get('Content-Type')}")
    print(f"Time taken to fetch data: {response.elapsed.total_seconds()} seconds")
    print(response.request)
    print(response.request.headers)
    print()

  # Find book elements
  soup = BeautifulSoup(response.text, "html.parser")
  book_elements = soup.find_all("article", class_ = "product_pod")

  book_data = []

  # Create a list of dictionaries containing one book's data each
  for item in book_elements:
    title_tag = item.h3.a
    price_tag = item.find("p", class_ = "price_color")
    in_stock_tag = item.find("p", class_ = "instock availability")
    rating_element = item.find("p", class_ = "star-rating")

    # Extract rating from class name
    rating = rating_element.get("class")[1] if rating_element else "N/A"

    # Extract title and link from attribute
    link = title_tag["href"] if title_tag else "N/A"
    title = title_tag["title"] if title_tag else "Unknown Title"

    # Extract tags if they exist
    price = price_tag.text.strip() if price_tag else "N/A"
    in_stock = in_stock_tag.text.strip() if in_stock_tag else "N/A"

    book_data.append({
        "Title": title,
        "Price": price,
        "Availability": in_stock,
        "Rating": rating,
        "Link": link
    })

  return book_data


def main():
  # JSONL contains individual, independent JSON objects on each line, making it superior for large datasets
  OUTPUT_FILE = "book_info.jsonl"

  with open(OUTPUT_FILE, "w") as file:
    page_number = 1
    while True:
      try:
        book_info = extract_book_info(page_number)
        if book_info:
          for book in book_info:
            # Add newline as this is a JSON lines file
            # ensure_ascii is False to keep the currency symbol as it is
            file.write(json.dumps(book, ensure_ascii = False) + "\n")

        # Avoid bombarding the server with numerous requests in a second
        # Use random.uniform(1, 2) so that a request hits the server at random intervals
        #wait_time = random.uniform(1, 2)
        #time.sleep(wait_time)

      except Exception as e:
        print(f"Failed to process page {page_number}. Error {e}.")

      if page_number == 50:
        break

      page_number += 1
      print(f"Searching page {page_number} now.")

  print(f"Data saved to {OUTPUT_FILE}.")

if __name__ == "__main__":
  main()

Print the first entry of the JSONL file.

In [ ]:
with open("book_info.jsonl") as file:
  firstline = file.readline()
  first_entry = json.loads(firstline)
  print(first_entry)

  # Reset pointer back to the start
  file.seek(0)

  lines = file.readlines()
  print(f"Number of lines: {len(lines)}")